In [7]:
# Install the required database orchestration libraries
# Using compatible 0.2.x versions and the latest stable Google GenAI integration
!pip install -q -U \
    langchain>=0.2.0 \
    langchain-core>=0.2.0 \
    langchain-community>=0.2.0 \
    langchain-google-genai \
    sqlalchemy>=2.0.30

print("[System] SQL and Agent environments installed successfully.")

[System] SQL and Agent environments installed successfully.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
#Boostrapping the corporate Database
import sqlite3
import pandas as pd 

print("=" * 60)
print("INITIATING TEXT-TO-SQL AGENT ORCHESTRATION")
print("=" * 60)


print("[System] Bootstraping local SQL-lite Data Warehouse...")

#Connect to SQL lite
conn = sqlite3.connect("corporate_warehouse.db")
cursor = conn.cursor()

# The DDL : Creatubg te executive_financials table 
cursor.execute('''
               CREATE TABLE IF NOT EXISTS executive_financials (
                    transaction_id TEXT PRIMARY KEY,
                    department_name TEXT,
                    fiscal_quarter TEXT,
                    revenue_usd REAL,
                    operational_costs REAL,
                    approval_status TEXT
                )
               ''')

# Injecting the mock data 
mock_data = [
    ('TXN-001', 'Engineering', 'Q3', 1500000.00, 800000.00, 'APPROVED'),
    ('TXN-002', 'Marketing', 'Q3', 450000.00, 600000.00, 'APPROVED'), # Note the loss here
    ('TXN-003', 'Sales', 'Q3', 3200000.00, 400000.00, 'APPROVED'),
    ('TXN-004', 'Engineering', 'Q4', 2100000.00, 950000.00, 'PENDING'),
    ('TXN-005', 'HR', 'Q4', 0.00, 120000.00, 'APPROVED')
]

cursor.executemany('''
                   INSERT OR IGNORE INTO executive_financials
                   VALUES (?,?,?,?,?,?)
                   ''', mock_data)

conn.commit()
conn.close()


print("[System Database 'corporate_warehouse.db' is online and populated]")

INITIATING TEXT-TO-SQL AGENT ORCHESTRATION
[System] Bootstraping local SQL-lite Data Warehouse...
[System Database 'corporate_warehouse.db' is online and populated]


In [2]:
# Wiring the SQL Agent 
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.agent_toolkits import create_sql_agent
import os
from pathlib import Path
from dotenv import load_dotenv

C:\Users\lenovo\AppData\Local\Temp\ipykernel_10716\2685269455.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase
c:\Users\Public\Documents\ai-foundations-lab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))



API key loaded: True


In [4]:
# Connect LangChain to the SQLite Database 

db = SQLDatabase.from_uri("sqlite:///corporate_warehouse.db")

# Initialize the AI compiler (Gemini 2.5 Flash)
# We set temperature to 0.0 because SQL requires strict determinism, not creativity. 
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", temperature = 0.0)

print("[System] Connecting AI Agent to the Database Schema...")

#Create the Autonomous SQL Agent 

sql_agent  = create_sql_agent(
    llm = llm, 
    db =db, 
    agent_type= "zero-shot-react-description", 
    verbose = True, 
    max_iterations= 30 # Circuit Breaker from part 4!
)

print("[System] Agent online. Ready for natural language queries.\n")

[System] Connecting AI Agent to the Database Schema...
[System] Agent online. Ready for natural language queries.



In [5]:
# The Execution Stress Test 

executive_query = ( 
    "Calculate the total net profit (revenue minus costs) across all departments "
    "for Q3. Only include approved transactions"
)

print("="*40)
print(f"User Query: '{executive_query}'")
print("="*40)

final_answer = sql_agent.invoke({"input": executive_query}, handle_parsing_errors=True)

print("\n" + "=" *40)
print(" FINAL SYNTHESIZED ANSWER ")
print("="*40)
print(final_answer['output'])


User Query: 'Calculate the total net profit (revenue minus costs) across all departments for Q3. Only include approved transactions'


> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input:executive_financialsI need to examine the schema of the `executive_financials` table to find columns related to revenue, costs, department, quarter, and transaction approval status.
Action: sql_db_schema
Action Input: executive_financials
CREATE TABLE executive_financials (
	transaction_id TEXT, 
	department_name TEXT, 
	fiscal_quarter TEXT, 
	revenue_usd REAL, 
	operational_costs REAL, 
	approval_status TEXT, 
	PRIMARY KEY (transaction_id)
)

/*
3 rows from executive_financials table:
transaction_id	department_name	fiscal_quarter	revenue_usd	operational_costs	approval_status
TXN-001	Engineering	Q3	1500000.0	800000.0	APPROVED
TXN-002	Marketing	Q3	450000.0	600000.0	APPROVED
TXN-003	Sales	Q3	3200000.0	400000.0	APPROVED
*/I have identified the `executive_financials` table an